# Intrusion Detection System — Comparison Models (RF & XGBoost)
### Dataset: CICIDS-2017  ·  Runs on **Kaggle** (GPU enabled for XGBoost)

## Role of this notebook

**These are reference implementations, not competitors.** Our methodological baseline is the from-scratch **logistic / softmax regression** — that's the simple model the scratch **MLP** was meant to beat (and did, by +0.087 binary F1 and +0.091 multi-class macro-F1).

The purpose of this notebook is different: **how close does our from-scratch NumPy implementation get to well-tuned production-grade libraries?** We train **Random Forest** (sklearn) and **XGBoost** (with GPU acceleration) on the same data, same evaluation set, same SMOTE-vs-class-weight comparison. The output is a clean 4-model × 2-task table that places the scratch results in context for the report.

## Pipeline

| # | Section |
|---|---------|
| 1 | Setup, Kaggle paths, report helpers |
| 2 | Load data — both training strategies, shared real-world test |
| 3 | Random Forest — small grid, then train both strategies on the best config |
| 4 | XGBoost (GPU) — same structure as RF |
| 5 | **The matrix:** 4 models × 2 tasks comparison + per-class breakdown |
| 6 | Save weights + metrics + figures |

Both tasks (binary + multi-class) are done in this single notebook so the comparison table lives in one place.

## Kaggle setup
1. Attach the FeatureSelection output dataset (same one used by all other modelling notebooks).
2. **Settings → Accelerator: GPU** (T4 or P100). XGBoost's GPU mode needs this. RF is CPU-only.
3. Set `IN_DIR` below to the dataset's mount path.
4. No `pip install` needed — RF and XGBoost ship with Kaggle's default image.

## 1. Imports, Config & Report Helpers

In [ ]:
import os, json, time, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CLASS_NAMES = ['BENIGN', 'Bot/Infiltration', 'Brute Force', 'DDoS',
               'DoS', 'PortScan', 'Web Attack']
N_CLASSES   = len(CLASS_NAMES)

# ── Kaggle paths ───────────────────────────────────────────────────
# IN_DIR : Kaggle Dataset holding the FeatureSelection output. EDIT to your mount path.
IN_DIR      = '/kaggle/input/ids-selected'
OUT_DIR     = '/kaggle/working'
FIGURES_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── report helpers ─────────────────────────────────────────────────
_report_lines = []

def _log(text=''):
    _report_lines.append(str(text))
    print(text)

def _savefig(name, fig=None):
    path = os.path.join(FIGURES_DIR, name)
    (fig or plt).savefig(path, dpi=130, bbox_inches='tight')
    return path

def write_report():
    path = os.path.join(OUT_DIR, 'Modeling_Comparison_Report.txt')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(_report_lines))
    print(f'\nReport saved -> {path}')

# detect GPU
try:
    import subprocess
    _ = subprocess.check_output(['nvidia-smi', '-L']).decode()
    GPU_AVAILABLE = True
except Exception:
    GPU_AVAILABLE = False

_log('=' * 70)
_log('COMPARISON-MODELS REPORT  —  CICIDS-2017')
_log('Random Forest (sklearn) + XGBoost (GPU if available)')
_log(f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
_log(f'GPU available : {GPU_AVAILABLE}')
_log('=' * 70)
print('\nSetup complete.')
print('  Reading from :', IN_DIR)
print('  Writing to   :', OUT_DIR)
print('  GPU mode     :', GPU_AVAILABLE)

## 2. Load Data — Both Imbalance Strategies, Both Tasks
Same FeatureSelection output dataset that the four scratch notebooks use. Loads everything once and reuses it across all four model/task combinations.

In [ ]:
train_path        = os.path.join(IN_DIR, 'train_selected.parquet')
smote_bin_path    = os.path.join(IN_DIR, 'train_binary_smote_selected.parquet')
smote_multi_path  = os.path.join(IN_DIR, 'train_multi_smote_selected.parquet')
test_path         = os.path.join(IN_DIR, 'test_selected.parquet')
feat_path         = os.path.join(IN_DIR, 'selected_features.json')

for p in [train_path, smote_bin_path, smote_multi_path, test_path, feat_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f'{p} not found. Set IN_DIR to the Kaggle mount path of your FeatureSelection output.')

with open(feat_path) as f:
    selected_features = json.load(f)['selected_features']

train_df       = pd.read_parquet(train_path)
smote_bin_df   = pd.read_parquet(smote_bin_path)
smote_multi_df = pd.read_parquet(smote_multi_path)
test_df        = pd.read_parquet(test_path)

# real-distribution training matrix (used for class-weight strategy + both tasks)
X_train_full  = train_df[selected_features].values.astype(np.float32)
y_bin_full    = train_df['label_binary'].values.astype(np.int64)
y_multi_full  = train_df['label_multi'].values.astype(np.int64)

# SMOTE training matrices (one per task)
X_smote_bin   = smote_bin_df[selected_features].values.astype(np.float32)
y_smote_bin   = smote_bin_df['label_binary'].values.astype(np.int64)
X_smote_multi = smote_multi_df[selected_features].values.astype(np.float32)
y_smote_multi = smote_multi_df['label_multi'].values.astype(np.int64)

# test (shared, real-world distribution)
X_test        = test_df[selected_features].values.astype(np.float32)
y_test_bin    = test_df['label_binary'].values.astype(np.int64)
y_test_multi  = test_df['label_multi'].values.astype(np.int64)

# tuning splits (carved from real training data; stratified)
X_tr_b, X_val_b, y_tr_b, y_val_b = train_test_split(
    X_train_full, y_bin_full, test_size=0.20,
    random_state=RANDOM_SEED, stratify=y_bin_full)
X_tr_m, X_val_m, y_tr_m, y_val_m = train_test_split(
    X_train_full, y_multi_full, test_size=0.20,
    random_state=RANDOM_SEED, stratify=y_multi_full)

n_features = X_train_full.shape[1]

_log('')
_log('── SECTION 2 : DATA LOADED ────────────────────────────────')
_log(f'  Features                 : {n_features}')
_log(f'  Real train               : {X_train_full.shape[0]:,} rows')
_log(f'  Binary SMOTE train       : {X_smote_bin.shape[0]:,} rows')
_log(f'  Multi-class SMOTE train  : {X_smote_multi.shape[0]:,} rows')
_log(f'  Test (held out, shared)  : {X_test.shape[0]:,} rows')
print('\nReady for Random Forest (CPU) and XGBoost (GPU).')

## 3. Random Forest
Small hyperparameter sweep over the few knobs that matter for tree ensembles on this data. We don't tune as deeply as the scratch models — RF is robust enough that a few reasonable settings cover the range.

| Knob | Grid | Note |
|---|---|---|
| `n_estimators` | 200 | Single value — more trees is monotonically better but slower; 200 is the standard "this is enough" point |
| `max_depth` | None, 20 | unrestricted vs constrained |
| `min_samples_leaf` | 1, 5 | overfit guard |

That's 4 fits per task, ranked by the task's primary metric. Class imbalance is handled by `class_weight='balanced'` for strategy A (matches the scratch LR/MLP approach). Strategy B uses the SMOTE training matrix.

In [ ]:
# ── helpers ──
def evaluate_binary(model, X, y):
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba(X)[:, 1]
    else:
        prob = model.predict(X).astype(float)
    pred = (prob >= 0.5).astype(int)
    return {
        'accuracy':  accuracy_score(y, pred),
        'precision': precision_score(y, pred),
        'recall':    recall_score(y, pred),
        'f1':        f1_score(y, pred),
        'roc_auc':   roc_auc_score(y, prob),
    }, pred, prob

def evaluate_multi(model, X, y):
    pred = model.predict(X)
    return {
        'accuracy':    accuracy_score(y, pred),
        'macro_f1':    f1_score(y, pred, average='macro'),
        'weighted_f1': f1_score(y, pred, average='weighted'),
        'macro_prec':  precision_score(y, pred, average='macro'),
        'macro_rec':   recall_score(y, pred, average='macro'),
    }, pred

def clean_max_depth(v):
    """pd.DataFrame stores None as NaN in numeric columns — restore it for sklearn."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    return int(v)

# ── BINARY RANDOM FOREST: grid search on tuning split ──
_log('')
_log('── SECTION 3a : RANDOM FOREST (BINARY) ────────────────────')
_log('  Grid search on the validation split...')
rf_grid_b = [
    {'max_depth': None, 'min_samples_leaf': 1},
    {'max_depth': None, 'min_samples_leaf': 5},
    {'max_depth': 20,   'min_samples_leaf': 1},
    {'max_depth': 20,   'min_samples_leaf': 5},
]
rf_b_results = []
for cfg in rf_grid_b:
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_SEED,
                                class_weight='balanced', **cfg)
    rf.fit(X_tr_b, y_tr_b)
    yv = rf.predict(X_val_b)
    row = {**cfg,
           'val_f1': f1_score(y_val_b, yv),
           'val_auc': roc_auc_score(y_val_b, rf.predict_proba(X_val_b)[:, 1]),
           'fit_seconds': time.time() - t0}
    rf_b_results.append(row)
    _log(f'  max_depth={cfg["max_depth"]!s:<5} min_leaf={cfg["min_samples_leaf"]:<2} -> '
         f'F1={row["val_f1"]:.4f}  AUC={row["val_auc"]:.4f}  ({row["fit_seconds"]:.1f}s)')

rf_b_df = pd.DataFrame(rf_b_results).sort_values('val_f1', ascending=False).reset_index(drop=True)
best_rf_b = rf_b_df.iloc[0]
best_rf_b_cfg = {'max_depth': clean_max_depth(best_rf_b['max_depth']),
                 'min_samples_leaf': int(best_rf_b['min_samples_leaf'])}
_log('')
_log(f'  BEST: max_depth={best_rf_b_cfg["max_depth"]}, '
     f'min_samples_leaf={best_rf_b_cfg["min_samples_leaf"]}  '
     f'(val F1={best_rf_b["val_f1"]:.4f})')

# ── BINARY: retrain best both ways, evaluate on test ──
_log('')
_log('  [A] class_weight=balanced on real training data ...')
t0 = time.time()
rf_b_A = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_SEED,
                                class_weight='balanced', **best_rf_b_cfg)
rf_b_A.fit(X_train_full, y_bin_full)
rf_b_metrics_A, rf_b_pred_A, rf_b_prob_A = evaluate_binary(rf_b_A, X_test, y_test_bin)
_log(f'      done in {time.time()-t0:.1f}s')

_log('  [B] SMOTE training data, no class_weight ...')
t0 = time.time()
rf_b_B = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_SEED,
                                **best_rf_b_cfg)
rf_b_B.fit(X_smote_bin, y_smote_bin)
rf_b_metrics_B, rf_b_pred_B, rf_b_prob_B = evaluate_binary(rf_b_B, X_test, y_test_bin)
_log(f'      done in {time.time()-t0:.1f}s')

rf_b_cmp = pd.DataFrame({'A_class_weight': rf_b_metrics_A, 'B_smote': rf_b_metrics_B})
rf_b_cmp['delta'] = rf_b_cmp['B_smote'] - rf_b_cmp['A_class_weight']
_log('')
_log('  TEST-SET RESULTS:')
_log(rf_b_cmp.to_string())
display(rf_b_cmp)

In [ ]:
# ── MULTI-CLASS RANDOM FOREST: grid search ──
_log('')
_log('── SECTION 3b : RANDOM FOREST (MULTI-CLASS) ───────────────')
_log('  Grid search on the validation split...')
rf_m_results = []
for cfg in rf_grid_b:           # reuse same grid; structure differs only in target
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_SEED,
                                class_weight='balanced', **cfg)
    rf.fit(X_tr_m, y_tr_m)
    yv = rf.predict(X_val_m)
    row = {**cfg,
           'val_macro_f1':    f1_score(y_val_m, yv, average='macro'),
           'val_weighted_f1': f1_score(y_val_m, yv, average='weighted'),
           'val_accuracy':    accuracy_score(y_val_m, yv),
           'fit_seconds':     time.time() - t0}
    rf_m_results.append(row)
    _log(f'  max_depth={cfg["max_depth"]!s:<5} min_leaf={cfg["min_samples_leaf"]:<2} -> '
         f'macroF1={row["val_macro_f1"]:.4f}  acc={row["val_accuracy"]:.4f}  '
         f'({row["fit_seconds"]:.1f}s)')

rf_m_df = pd.DataFrame(rf_m_results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
best_rf_m = rf_m_df.iloc[0]
best_rf_m_cfg = {'max_depth': clean_max_depth(best_rf_m['max_depth']),
                 'min_samples_leaf': int(best_rf_m['min_samples_leaf'])}
_log('')
_log(f'  BEST: max_depth={best_rf_m_cfg["max_depth"]}, '
     f'min_samples_leaf={best_rf_m_cfg["min_samples_leaf"]}  '
     f'(val macro-F1={best_rf_m["val_macro_f1"]:.4f})')

# retrain both strategies on full data
_log('')
_log('  [A] class_weight=balanced on real training data ...')
t0 = time.time()
rf_m_A = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_SEED,
                                class_weight='balanced', **best_rf_m_cfg)
rf_m_A.fit(X_train_full, y_multi_full)
rf_m_metrics_A, rf_m_pred_A = evaluate_multi(rf_m_A, X_test, y_test_multi)
_log(f'      done in {time.time()-t0:.1f}s')

_log('  [B] SMOTE training data, no class_weight ...')
t0 = time.time()
rf_m_B = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_SEED,
                                **best_rf_m_cfg)
rf_m_B.fit(X_smote_multi, y_smote_multi)
rf_m_metrics_B, rf_m_pred_B = evaluate_multi(rf_m_B, X_test, y_test_multi)
_log(f'      done in {time.time()-t0:.1f}s')

rf_m_cmp = pd.DataFrame({'A_class_weight': rf_m_metrics_A, 'B_smote': rf_m_metrics_B})
rf_m_cmp['delta'] = rf_m_cmp['B_smote'] - rf_m_cmp['A_class_weight']
_log('')
_log('  TEST-SET RESULTS:')
_log(rf_m_cmp.to_string())
display(rf_m_cmp)

# pick the RF winners (used in Section 5)
rf_b_winner = ('A', rf_b_metrics_A, rf_b_pred_A, rf_b_prob_A) if rf_b_metrics_A['f1'] >= rf_b_metrics_B['f1'] else ('B', rf_b_metrics_B, rf_b_pred_B, rf_b_prob_B)
rf_m_winner = ('A', rf_m_metrics_A, rf_m_pred_A) if rf_m_metrics_A['macro_f1'] >= rf_m_metrics_B['macro_f1'] else ('B', rf_m_metrics_B, rf_m_pred_B)
_log('')
_log(f'  RF binary winner      : strategy {rf_b_winner[0]}  (F1 = {rf_b_winner[1]["f1"]:.4f})')
_log(f'  RF multi-class winner : strategy {rf_m_winner[0]}  (macro-F1 = {rf_m_winner[1]["macro_f1"]:.4f})')

## 4. XGBoost (GPU)
Gradient-boosted trees, the production gold standard for tabular data. Uses `tree_method='hist', device='cuda'` for GPU acceleration — this is the one model in the whole project that actually benefits from Kaggle's GPU.

Small grid over the two most impactful knobs:

| Knob | Grid |
|---|---|
| `max_depth` | 6, 10 |
| `learning_rate` | 0.1, 0.3 |
| `n_estimators` | 400 (with early stopping safety) |

Strategy A uses `scale_pos_weight` (binary) or `sample_weight` (multi-class) for class imbalance. Strategy B uses SMOTE training data.

In [ ]:
XGB_DEVICE = 'cuda' if GPU_AVAILABLE else 'cpu'
XGB_COMMON = dict(tree_method='hist', device=XGB_DEVICE,
                  n_estimators=400, random_state=RANDOM_SEED, n_jobs=-1,
                  verbosity=0)

# ── BINARY XGBOOST: grid search ──
_log('')
_log(f'── SECTION 4a : XGBOOST (BINARY)  device={XGB_DEVICE} ──────')
xgb_grid = [
    {'max_depth': 6,  'learning_rate': 0.1},
    {'max_depth': 6,  'learning_rate': 0.3},
    {'max_depth': 10, 'learning_rate': 0.1},
    {'max_depth': 10, 'learning_rate': 0.3},
]

# scale_pos_weight = neg/pos  -> binary class balance
spw = (y_tr_b == 0).sum() / (y_tr_b == 1).sum()

xgb_b_results = []
for cfg in xgb_grid:
    t0 = time.time()
    m = xgb.XGBClassifier(scale_pos_weight=spw, **XGB_COMMON, **cfg)
    m.fit(X_tr_b, y_tr_b)
    yv = m.predict(X_val_b)
    prob = m.predict_proba(X_val_b)[:, 1]
    row = {**cfg,
           'val_f1':  f1_score(y_val_b, yv),
           'val_auc': roc_auc_score(y_val_b, prob),
           'fit_seconds': time.time() - t0}
    xgb_b_results.append(row)
    _log(f'  max_depth={cfg["max_depth"]:<2} lr={cfg["learning_rate"]:<4} -> '
         f'F1={row["val_f1"]:.4f}  AUC={row["val_auc"]:.4f}  ({row["fit_seconds"]:.1f}s)')

xgb_b_df = pd.DataFrame(xgb_b_results).sort_values('val_f1', ascending=False).reset_index(drop=True)
best_xgb_b = xgb_b_df.iloc[0]
best_xgb_b_cfg = {'max_depth': int(best_xgb_b['max_depth']),
                  'learning_rate': float(best_xgb_b['learning_rate'])}
_log('')
_log(f'  BEST: max_depth={best_xgb_b_cfg["max_depth"]}, '
     f'learning_rate={best_xgb_b_cfg["learning_rate"]}  '
     f'(val F1={best_xgb_b["val_f1"]:.4f})')

# retrain both strategies on full data
spw_full = (y_bin_full == 0).sum() / (y_bin_full == 1).sum()
_log('')
_log('  [A] scale_pos_weight on real training data ...')
t0 = time.time()
xgb_b_A = xgb.XGBClassifier(scale_pos_weight=spw_full, **XGB_COMMON, **best_xgb_b_cfg)
xgb_b_A.fit(X_train_full, y_bin_full)
xgb_b_metrics_A, xgb_b_pred_A, xgb_b_prob_A = evaluate_binary(xgb_b_A, X_test, y_test_bin)
_log(f'      done in {time.time()-t0:.1f}s')

_log('  [B] SMOTE training data, scale_pos_weight=1 ...')
t0 = time.time()
xgb_b_B = xgb.XGBClassifier(scale_pos_weight=1.0, **XGB_COMMON, **best_xgb_b_cfg)
xgb_b_B.fit(X_smote_bin, y_smote_bin)
xgb_b_metrics_B, xgb_b_pred_B, xgb_b_prob_B = evaluate_binary(xgb_b_B, X_test, y_test_bin)
_log(f'      done in {time.time()-t0:.1f}s')

xgb_b_cmp = pd.DataFrame({'A_class_weight': xgb_b_metrics_A, 'B_smote': xgb_b_metrics_B})
xgb_b_cmp['delta'] = xgb_b_cmp['B_smote'] - xgb_b_cmp['A_class_weight']
_log('')
_log('  TEST-SET RESULTS:')
_log(xgb_b_cmp.to_string())
display(xgb_b_cmp)

In [ ]:
# ── MULTI-CLASS XGBOOST: grid search ──
_log('')
_log(f'── SECTION 4b : XGBOOST (MULTI-CLASS)  device={XGB_DEVICE} ──')

# multi-class sample weights = inverse class frequency on the tuning split
def inverse_freq_weights(y, K):
    cc = np.bincount(y, minlength=K).astype(float)
    cw = len(y) / (K * np.maximum(cc, 1))
    return cw[y]

sw_tr_m = inverse_freq_weights(y_tr_m, N_CLASSES)

xgb_m_results = []
for cfg in xgb_grid:
    t0 = time.time()
    m = xgb.XGBClassifier(objective='multi:softprob', num_class=N_CLASSES,
                          **XGB_COMMON, **cfg)
    m.fit(X_tr_m, y_tr_m, sample_weight=sw_tr_m)
    yv = m.predict(X_val_m)
    row = {**cfg,
           'val_macro_f1':    f1_score(y_val_m, yv, average='macro'),
           'val_weighted_f1': f1_score(y_val_m, yv, average='weighted'),
           'val_accuracy':    accuracy_score(y_val_m, yv),
           'fit_seconds':     time.time() - t0}
    xgb_m_results.append(row)
    _log(f'  max_depth={cfg["max_depth"]:<2} lr={cfg["learning_rate"]:<4} -> '
         f'macroF1={row["val_macro_f1"]:.4f}  acc={row["val_accuracy"]:.4f}  '
         f'({row["fit_seconds"]:.1f}s)')

xgb_m_df = pd.DataFrame(xgb_m_results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
best_xgb_m = xgb_m_df.iloc[0]
best_xgb_m_cfg = {'max_depth': int(best_xgb_m['max_depth']),
                  'learning_rate': float(best_xgb_m['learning_rate'])}
_log('')
_log(f'  BEST: max_depth={best_xgb_m_cfg["max_depth"]}, '
     f'learning_rate={best_xgb_m_cfg["learning_rate"]}  '
     f'(val macro-F1={best_xgb_m["val_macro_f1"]:.4f})')

# retrain both strategies on full data
sw_full = inverse_freq_weights(y_multi_full, N_CLASSES)
_log('')
_log('  [A] inverse-frequency sample weights on real training data ...')
t0 = time.time()
xgb_m_A = xgb.XGBClassifier(objective='multi:softprob', num_class=N_CLASSES,
                            **XGB_COMMON, **best_xgb_m_cfg)
xgb_m_A.fit(X_train_full, y_multi_full, sample_weight=sw_full)
xgb_m_metrics_A, xgb_m_pred_A = evaluate_multi(xgb_m_A, X_test, y_test_multi)
_log(f'      done in {time.time()-t0:.1f}s')

_log('  [B] SMOTE training data, uniform weights ...')
t0 = time.time()
xgb_m_B = xgb.XGBClassifier(objective='multi:softprob', num_class=N_CLASSES,
                            **XGB_COMMON, **best_xgb_m_cfg)
xgb_m_B.fit(X_smote_multi, y_smote_multi)
xgb_m_metrics_B, xgb_m_pred_B = evaluate_multi(xgb_m_B, X_test, y_test_multi)
_log(f'      done in {time.time()-t0:.1f}s')

xgb_m_cmp = pd.DataFrame({'A_class_weight': xgb_m_metrics_A, 'B_smote': xgb_m_metrics_B})
xgb_m_cmp['delta'] = xgb_m_cmp['B_smote'] - xgb_m_cmp['A_class_weight']
_log('')
_log('  TEST-SET RESULTS:')
_log(xgb_m_cmp.to_string())
display(xgb_m_cmp)

xgb_b_winner = ('A', xgb_b_metrics_A, xgb_b_pred_A, xgb_b_prob_A) if xgb_b_metrics_A['f1'] >= xgb_b_metrics_B['f1'] else ('B', xgb_b_metrics_B, xgb_b_pred_B, xgb_b_prob_B)
xgb_m_winner = ('A', xgb_m_metrics_A, xgb_m_pred_A) if xgb_m_metrics_A['macro_f1'] >= xgb_m_metrics_B['macro_f1'] else ('B', xgb_m_metrics_B, xgb_m_pred_B)
_log('')
_log(f'  XGB binary winner      : strategy {xgb_b_winner[0]}  (F1 = {xgb_b_winner[1]["f1"]:.4f})')
_log(f'  XGB multi-class winner : strategy {xgb_m_winner[0]}  (macro-F1 = {xgb_m_winner[1]["macro_f1"]:.4f})')

## 5. The 4-Model × 2-Task Comparison Matrix
All four models on both tasks, on the same held-out test set, using each model's winning imbalance strategy. The scratch-LR and scratch-MLP numbers are reproduced from the previous notebooks for direct comparison.

**This is the headline visualisation of the report.** Read the columns left to right: methodological baseline → proposed scratch model → reference RF → reference XGBoost.

In [ ]:
# ── scratch-model results (reproduced from earlier notebooks) ──
scratch_results = {
    'binary': {
        'Scratch LR':  {'accuracy': 0.9665, 'precision': 0.8501, 'recall': 0.9680,
                        'f1': 0.9052, 'roc_auc': 0.9932},
        'Scratch MLP': {'accuracy': 0.9975, 'precision': 0.9892, 'recall': 0.9961,
                        'f1': 0.9926, 'roc_auc': 0.9999},
    },
    'multi': {
        'Scratch LR':  {'accuracy': 0.9704, 'macro_f1': 0.7527, 'weighted_f1': 0.9798,
                        'macro_prec': 0.7142, 'macro_rec': 0.9771},
        'Scratch MLP': {'accuracy': 0.9924, 'macro_f1': 0.8439, 'weighted_f1': 0.9944,
                        'macro_prec': 0.8070, 'macro_rec': 0.9878},
    },
}

# ── binary matrix ──
binary_matrix = pd.DataFrame({
    'Scratch LR':  scratch_results['binary']['Scratch LR'],
    'Scratch MLP': scratch_results['binary']['Scratch MLP'],
    'Random Forest': rf_b_winner[1],
    'XGBoost':       xgb_b_winner[1],
})

# ── multi-class matrix ──
multi_matrix = pd.DataFrame({
    'Scratch LR':  scratch_results['multi']['Scratch LR'],
    'Scratch MLP': scratch_results['multi']['Scratch MLP'],
    'Random Forest': rf_m_winner[1],
    'XGBoost':       xgb_m_winner[1],
})

_log('')
_log('=' * 70)
_log('SECTION 5 : 4-MODEL × 2-TASK COMPARISON')
_log('=' * 70)
_log('')
_log('BINARY (test set, real-world distribution):')
_log(binary_matrix.to_string())
print('BINARY:'); display(binary_matrix)

_log('')
_log('MULTI-CLASS (test set, real-world distribution):')
_log(multi_matrix.to_string())
print('\nMULTI-CLASS:'); display(multi_matrix)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(22, 12))
model_colors = {'Scratch LR': '#FFA000', 'Scratch MLP': '#7B1FA2',
                'Random Forest': '#388E3C', 'XGBoost': '#1976D2'}
models = list(binary_matrix.columns)

# (a) binary metric bars
ax = axes[0, 0]
bin_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
xx = np.arange(len(bin_metrics))
width = 0.2
for i, mdl in enumerate(models):
    vals = [binary_matrix.loc[m, mdl] for m in bin_metrics]
    ax.bar(xx + (i - 1.5) * width, vals, width, label=mdl, color=model_colors[mdl])
ax.set_xticks(xx); ax.set_xticklabels(bin_metrics, rotation=15)
ax.set_ylim(0.7, 1.02)
ax.set_title('BINARY — all 4 models on the test set', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)

# (b) multi-class metric bars
ax = axes[0, 1]
mul_metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_prec', 'macro_rec']
xx = np.arange(len(mul_metrics))
for i, mdl in enumerate(models):
    vals = [multi_matrix.loc[m, mdl] for m in mul_metrics]
    ax.bar(xx + (i - 1.5) * width, vals, width, label=mdl, color=model_colors[mdl])
ax.set_xticks(xx); ax.set_xticklabels(mul_metrics, rotation=15)
ax.set_ylim(0.55, 1.02)
ax.set_title('MULTI-CLASS — all 4 models on the test set', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)

# (c) heatmap: binary matrix
ax = axes[1, 0]
sns.heatmap(binary_matrix, annot=True, fmt='.4f', cmap='YlGnBu',
            ax=ax, vmin=0.7, vmax=1.0, cbar_kws={'label': 'score'})
ax.set_title('Binary — value table', fontweight='bold')

# (d) heatmap: multi-class matrix
ax = axes[1, 1]
sns.heatmap(multi_matrix, annot=True, fmt='.4f', cmap='YlGnBu',
            ax=ax, vmin=0.55, vmax=1.0, cbar_kws={'label': 'score'})
ax.set_title('Multi-class — value table', fontweight='bold')

plt.suptitle('4 MODELS × 2 TASKS — Test-set Performance Matrix',
             fontsize=15, fontweight='bold')
plt.tight_layout()
_savefig('01_comparison_matrix.png', fig)
plt.show()

In [ ]:
# ── per-class F1 for multi-class — the rare-class story ──
f1_LR_multi  = np.array([0.9823, 0.0727, 0.8616, 0.9966, 0.9571, 0.9900, 0.4084])  # softmax LR
f1_MLP_multi = np.array([0.9955, 0.2291, 0.9685, 0.9987, 0.9914, 0.9934, 0.7306])  # scratch MLP
f1_RF_multi  = f1_score(y_test_multi, rf_m_winner[2], average=None)
f1_XGB_multi = f1_score(y_test_multi, xgb_m_winner[2], average=None)

perclass_df = pd.DataFrame({
    'Scratch LR':    f1_LR_multi,
    'Scratch MLP':   f1_MLP_multi,
    'Random Forest': f1_RF_multi,
    'XGBoost':       f1_XGB_multi,
}, index=CLASS_NAMES)

_log('')
_log('  PER-CLASS F1 (multi-class, winning strategy per model):')
_log(perclass_df.to_string())
display(perclass_df)

# grouped bar
fig, ax = plt.subplots(figsize=(16, 6))
xx = np.arange(N_CLASSES)
for i, mdl in enumerate(models):
    ax.bar(xx + (i - 1.5) * 0.2, perclass_df[mdl].values, 0.2,
           label=mdl, color=model_colors[mdl])
ax.set_xticks(xx); ax.set_xticklabels(CLASS_NAMES, rotation=25, ha='right')
ax.set_ylabel('F1'); ax.set_ylim(0, 1.05)
ax.set_title('Per-Class F1 (Multi-class) — Where do the models disagree?',
             fontweight='bold')
ax.legend(loc='lower left', fontsize=10)
plt.tight_layout()
_savefig('02_perclass_f1.png', fig)
plt.show()

# heatmap version for the report
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(perclass_df.T, annot=True, fmt='.3f', cmap='YlGnBu',
            vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'F1'})
ax.set_title('Per-Class F1 Heatmap (Multi-class)', fontweight='bold')
plt.tight_layout()
_savefig('03_perclass_heatmap.png', fig)
plt.show()

## 6. Save Models, Metrics & Report

In [ ]:
# save the winning models
joblib.dump(rf_b_A if rf_b_winner[0] == 'A' else rf_b_B,
            os.path.join(OUT_DIR, 'rf_binary_best.joblib'))
joblib.dump(rf_m_A if rf_m_winner[0] == 'A' else rf_m_B,
            os.path.join(OUT_DIR, 'rf_multi_best.joblib'))
(xgb_b_A if xgb_b_winner[0] == 'A' else xgb_b_B).save_model(
    os.path.join(OUT_DIR, 'xgb_binary_best.json'))
(xgb_m_A if xgb_m_winner[0] == 'A' else xgb_m_B).save_model(
    os.path.join(OUT_DIR, 'xgb_multi_best.json'))

# comparison tables
binary_matrix.to_csv(os.path.join(OUT_DIR, 'comparison_matrix_binary.csv'))
multi_matrix.to_csv(os.path.join(OUT_DIR, 'comparison_matrix_multi.csv'))
perclass_df.to_csv(os.path.join(OUT_DIR, 'comparison_perclass_multi.csv'))

# combined metrics JSON
with open(os.path.join(OUT_DIR, 'comparison_metrics.json'), 'w') as f:
    json.dump({
        'binary': {
            'Scratch LR':    scratch_results['binary']['Scratch LR'],
            'Scratch MLP':   scratch_results['binary']['Scratch MLP'],
            'Random Forest': rf_b_winner[1],
            'XGBoost':       xgb_b_winner[1],
            'rf_winning_strategy':  rf_b_winner[0],
            'xgb_winning_strategy': xgb_b_winner[0],
        },
        'multi': {
            'Scratch LR':    scratch_results['multi']['Scratch LR'],
            'Scratch MLP':   scratch_results['multi']['Scratch MLP'],
            'Random Forest': rf_m_winner[1],
            'XGBoost':       xgb_m_winner[1],
            'rf_winning_strategy':  rf_m_winner[0],
            'xgb_winning_strategy': xgb_m_winner[0],
        },
    }, f, indent=2)

_log('')
_log('── SECTION 6 : FILES SAVED ────────────────────────────────')
for f in ['rf_binary_best.joblib', 'rf_multi_best.joblib',
          'xgb_binary_best.json', 'xgb_multi_best.json',
          'comparison_matrix_binary.csv', 'comparison_matrix_multi.csv',
          'comparison_perclass_multi.csv', 'comparison_metrics.json']:
    print(f'  {f}')
    _log(f'  {f}')

In [ ]:
_log('')
_log('=' * 70)
_log('SUMMARY  --  COMPARISON MODELS')
_log('=' * 70)
_log('')
_log('  Role of this notebook:')
_log('    Methodological baseline = Scratch LR (the simple model we beat to')
_log('    prove the MLP\'s non-linearity helps). RF + XGBoost are REFERENCE')
_log('    models that show how close the from-scratch implementations get')
_log('    to production-grade libraries.')
_log('')
_log('  BINARY TEST-SET MATRIX:')
_log(binary_matrix.to_string())
_log('')
_log('  MULTI-CLASS TEST-SET MATRIX:')
_log(multi_matrix.to_string())
_log('')
_log('  PER-CLASS F1 (multi-class):')
_log(perclass_df.to_string())
_log('')
_log('  Winning imbalance strategies:')
_log(f'    Random Forest  binary       : {rf_b_winner[0]}')
_log(f'    Random Forest  multi-class  : {rf_m_winner[0]}')
_log(f'    XGBoost        binary       : {xgb_b_winner[0]}')
_log(f'    XGBoost        multi-class  : {xgb_m_winner[0]}')

FIGURE_INDEX = [
    ('01_comparison_matrix.png',  '4 models x 2 tasks — bars + value heatmaps'),
    ('02_perclass_f1.png',        'Per-class F1 grouped bar (multi-class)'),
    ('03_perclass_heatmap.png',   'Per-class F1 heatmap (multi-class)'),
]
_log('')
_log('  Figures:')
for fname, desc in FIGURE_INDEX:
    _log(f'    {fname:<32} {desc}')

write_report()

print('\n' + '=' * 55)
print('COMPARISON MODELS COMPLETE')
print('=' * 55)
print('Binary F1:')
for c in binary_matrix.columns:
    print(f'  {c:<14} : {binary_matrix.loc["f1", c]:.4f}')
print('Multi-class macro-F1:')
for c in multi_matrix.columns:
    print(f'  {c:<14} : {multi_matrix.loc["macro_f1", c]:.4f}')
print(f'\nReport  -> {OUT_DIR}/Modeling_Comparison_Report.txt')
print(f'Figures -> {FIGURES_DIR}/  ({len(FIGURE_INDEX)} figures)')